# 02 — Preprocessing Pipeline
**Teammate Matcher | DTSC 2302**

This notebook executes and documents the full preprocessing pipeline defined in `src/preprocess.py`.  
Each step is shown with **before/after comparisons** and the rationale for every design decision.

Input:  `data/raw_survey_responses.csv`  
Output: `data/processed_survey_data.csv` (31 rows × 50 features)

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import (
    load_raw, clean, encode_availability, encode_ordinals,
    encode_onehot, encode_contributions, handle_missing,
    normalize, build_feature_sets, run_pipeline,
    YEAR_MAP, HOURS_MAP, ROLE_MAP, DEADLINE_MAP,
    CHECKIN_MAP, COLLAB_MAP, GPA_MAP,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
UNCC_GREEN = "#00703C"
UNCC_GOLD  = "#B3A369"
plt.rcParams["figure.dpi"] = 120

RAW_PATH = "../data/raw_survey_responses.csv"
OUT_PATH = "../data/processed_survey_data.csv"

---
## Step 1 — Load Raw Data

In [ ]:
df = load_raw(RAW_PATH)
print(f"Raw shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(2)

In [ ]:
print("All columns in raw CSV:")
for i, c in enumerate(df.columns):
    print(f"  [{i:02d}] {c}")

---
## Step 2 — Clean: Column Renaming & Artifact Removal

**What:** Rename 25 known columns to short internal names; drop artifact `Column 25` fields  
and the `Timestamp` column (not analytically useful).  
Normalize `course_code` to `"DTSC 2302"` — 11 free-text variants all mean the same course.

**Why:** Raw Google Forms headers are too long for use in code. The two `Column 25` fields  
are empty artifacts generated by Google Forms for checkbox questions.

In [ ]:
df_clean = clean(df)

print(f"After cleaning: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"Columns dropped: {df.shape[1] - df_clean.shape[1]}")
print("\nNew column names:")
for c in df_clean.columns:
    print(f"  {c}")

In [ ]:
# Verify course_code normalization
print("course_code unique values after clean:", df_clean["course_code"].unique())

---
## Step 3 — Availability Encoding

**What:** Split comma-separated `_days_raw` and `_times_raw` columns into 11 binary columns:  
- 7 day columns: `avail_mon` … `avail_sun`  
- 4 time columns: `avail_morning` … `avail_latenight`

**Why:** Two students' availability vectors can be directly compared via Jaccard similarity —  
a natural measure of scheduling overlap that informs the compatibility objective.

In [ ]:
print("BEFORE — raw availability columns:")
print(df_clean[["_days_raw","_times_raw"]].head(4).to_string())

In [ ]:
df_avail = encode_availability(df_clean)

avail_cols = [c for c in df_avail.columns if c.startswith("avail_")]
print(f"AFTER — {len(avail_cols)} binary availability columns:")
print(df_avail[avail_cols].head(4).to_string())

In [ ]:
# Proportion of respondents available on each slot
avail_means = df_avail[avail_cols].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(avail_means.index, avail_means.values * 100,
       color=UNCC_GREEN, edgecolor="white")
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, label="50% mark")
ax.set_title("% of Respondents Available per Slot", fontweight="bold")
ax.set_ylabel("%")
ax.tick_params(axis="x", rotation=30)
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

---
## Step 4 — Ordinal Encoding

**What:** Map ordered categorical responses to integers, preserving their natural ordering.

| Column | Encoding |
|---|---|
| `year` | Freshman=1 … Graduate=5 |
| `weekly_hours` | <3hrs=1, 3–5=2, 6–9=3, 10+=4 |
| `role_pref` | Follower=1, Specialist=2, Flexible=3, Leader=4 |
| `deadline_style` | Last-minute=1, Steady=2, Early=3 |
| `checkin_freq` | As needed=1, Weekly=2, Few/week=3, Daily=4 |
| `collab_style` | Independent=1, Mix=2, Close=3 |
| `gpa_band` | <2.5=1, 2.5–3.0=2, 3.0–3.5=3, 3.5–4.0=4; "Prefer not to say" → NaN |

**Why:** These variables have a clear semantic ordering — encoding them as integers allows  
distance-based algorithms to correctly treat, e.g., a Steady worker as "between"  
Last-minute and Early rather than in a completely separate category.

In [ ]:
ordinal_cols = ["year","weekly_hours","role_pref","deadline_style",
                "checkin_freq","collab_style","gpa_band"]

print("BEFORE — sample raw values:")
print(df_avail[ordinal_cols].head(4).to_string())

df_ord = encode_ordinals(df_avail)

print("\nAFTER — mapped to integers:")
print(df_ord[ordinal_cols].head(4).to_string())

In [ ]:
print("Ordinal encodings used:")
for name, mapping in [("year", YEAR_MAP), ("weekly_hours", HOURS_MAP),
                       ("role_pref", ROLE_MAP), ("deadline_style", DEADLINE_MAP),
                       ("checkin_freq", CHECKIN_MAP), ("collab_style", COLLAB_MAP),
                       ("gpa_band", GPA_MAP)]:
    print(f"\n  {name}:")
    for k, v in mapping.items():
        print(f"    {v} ← '{k}'")

---
## Step 5 — One-Hot Encoding

**What:** Expand nominal (unordered) categoricals into individual binary columns:
- `meeting_mode` → `meeting_inperson`, `meeting_remote`, `meeting_nopref`
- `comm_pref` → `comm_text`, `comm_email`, `comm_discord`, `comm_video`, `comm_inperson`
- `conflict_style` → `conflict_direct`, `conflict_private`, `conflict_natural`, `conflict_defer`
- `pain_point` → `pain_schedule`, `pain_workload`, `pain_conflict`, `pain_communication`, `pain_motivation`

**Why:** Meeting mode and communication channel have no meaningful ordering.  
One-hot encoding prevents the algorithm from treating `comm_discord` as  
"numerically between" `comm_text` and `comm_email`.

In [ ]:
print("BEFORE — nominal columns:")
print(df_ord[["meeting_mode","comm_pref","conflict_style","pain_point"]].head(4).to_string())

df_ohe = encode_onehot(df_ord)

ohe_cols = [c for c in df_ohe.columns if any(
    c.startswith(p) for p in ["meeting_","comm_","conflict_","pain_"])]
print(f"\nAFTER — {len(ohe_cols)} binary columns:")
print(df_ohe[ohe_cols].head(4).to_string())

---
## Step 6 — Contribution Multi-Select Encoding

**What:** Expand the comma-separated `_contrib_raw` column into 6 binary indicator columns:
`contrib_technical`, `contrib_creative`, `contrib_organization`,
`contrib_writing`, `contrib_morale`, `contrib_qa`.

**Why:** Unlike one-hot (exactly one category), contributions are multi-select.  
Each is an independent yes/no indicator — a student can score 1 on multiple columns.  
The "up to 2" limit was not enforced (16/31 selected more); we treat as binary indicators.

In [ ]:
df_contrib = encode_contributions(df_ohe)

contrib_cols = [c for c in df_contrib.columns if c.startswith("contrib_")]
print(f"Contribution columns ({len(contrib_cols)}):")
print(df_contrib[contrib_cols].head(6).to_string())
print("\nColumn sums (how many respondents selected each):")
print(df_contrib[contrib_cols].sum().to_string())

---
## Step 7 — Missing Value Handling

**What:**
1. Impute `gpa_band` NaN values with the **median** ordinal band
2. Drop any row where >20% of numeric features are missing

**Why median?** GPA is an ordinal variable (bands 1–4). Mean imputation on an ordinal  
variable can produce non-integer values that fall between valid categories.  
Median preserves the ordinal scale. With only 2 missing values out of 31, imputation  
has minimal impact on clustering.

In [ ]:
print(f"GPA missing before imputation: {df_contrib['gpa_band'].isna().sum()}")
print(f"GPA median (non-missing):      {df_contrib['gpa_band'].median()}")

df_filled = handle_missing(df_contrib)

print(f"\nRows before: {df_contrib.shape[0]}")
print(f"Rows after:  {df_filled.shape[0]}")
print(f"GPA missing after: {df_filled['gpa_band'].isna().sum()}")

---
## Step 8 — Min-Max Normalization

**What:** Scale all continuous and ordinal features to [0, 1].  
Binary columns (already 0/1) are **excluded** from scaling.

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

**Why:** Skill ratings (1–5) and ordinal year-in-school (1–5) have larger raw ranges  
than binary availability columns (0–1). Without normalization, high-variance features  
would dominate Euclidean distance calculations in K-Means, agglomerative, and the  
Hungarian cost matrix, effectively ignoring binary features.

In [ ]:
numeric_cols = df_filled.select_dtypes(include=[np.number]).columns.tolist()
binary_cols  = [c for c in numeric_cols if df_filled[c].isin([0,1]).all()]
scale_cols   = [c for c in numeric_cols if c not in binary_cols]

print(f"Total numeric columns:     {len(numeric_cols)}")
print(f"Already-binary (skip):     {len(binary_cols)}")
print(f"Columns to normalize:      {len(scale_cols)}")
print(f"\nColumns being normalized:")
for c in scale_cols:
    rng = f"[{df_filled[c].min():.1f}, {df_filled[c].max():.1f}]"
    print(f"  {c:<22} raw range {rng}")

In [ ]:
df_norm = normalize(df_filled)

print("\nAFTER normalization — sample of scaled columns:")
print(df_norm[scale_cols[:6]].describe().round(3))

In [ ]:
# ── Visual: before vs. after for skill_python ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(df_filled["skill_python"], bins=5, color=UNCC_GREEN,
             edgecolor="white", rwidth=0.85)
axes[0].set_title("skill_python — BEFORE normalization", fontweight="bold")
axes[0].set_xlabel("Raw rating (1–5)")

axes[1].hist(df_norm["skill_python"], bins=10, color=UNCC_GOLD,
             edgecolor="white", rwidth=0.85)
axes[1].set_title("skill_python — AFTER normalization", fontweight="bold")
axes[1].set_xlabel("Scaled [0, 1]")

for ax in axes:
    ax.set_ylabel("Count")
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

---
## Step 9 — Save Processed Data

In [ ]:
# Drop metadata (course_code is not a clustering feature)
save_df = df_norm.drop(columns=["course_code"], errors="ignore")
save_df.to_csv(OUT_PATH, index=False)
print(f"Saved: {save_df.shape[0]} rows × {save_df.shape[1]} features → {OUT_PATH}")
print(f"\nFinal columns:")
for c in save_df.columns:
    dtype_flag = "float" if save_df[c].dtype == float else "int"
    print(f"  {c:<26} [{dtype_flag}]")

---
## Step 10 — Feature Sets

Two feature sets are constructed for use by different model families:

| Feature Set | Contents | Model Family | Objective |
|---|---|---|---|
| `compatibility` | Availability (11) + work style (18) | K-Means, Agglomerative, Hungarian | Minimize intra-team friction |
| `complementarity` | Skills only (8) | GMM | Maximize skill diversity |
| `all_features` | Both sets combined (37) | PCA, evaluation | Feature importance analysis |

In [ ]:
feature_sets = build_feature_sets(save_df)

for name, fset in feature_sets.items():
    print(f"\n── {name} ({fset.shape[1]} features) ──")
    print(", ".join(fset.columns.tolist()))

In [ ]:
# ── Correlation heatmap of compatibility feature set ──────────────────────────
compat_df = feature_sets["compatibility"]

fig, ax = plt.subplots(figsize=(14, 11))
corr = compat_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".1f",
    cmap="RdYlGn", center=0, linewidths=0.3, linecolor="white",
    ax=ax, cbar_kws={"shrink": 0.7},
    annot_kws={"size": 7}
)
ax.set_title("Compatibility Feature Correlations", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Full Pipeline Run (Convenience)

The cell below re-runs the entire pipeline in one call via `run_pipeline()`.  
This is the entry point used by all downstream notebooks.

In [ ]:
df_final, fsets = run_pipeline(
    raw_path="../data/raw_survey_responses.csv",
    out_path="../data/processed_survey_data.csv"
)

print(f"\nReturned df_final: {df_final.shape}")
print(f"Feature sets: {list(fsets.keys())}")

---
## Preprocessing Summary

| Step | Action | Columns Before | Columns After |
|---|---|---|---|
| 1. Load | Read raw CSV | 27 | 27 |
| 2. Clean | Rename + drop artifact cols + normalize course_code | 27 | 25 |
| 3. Availability | Expand 2 checkbox cols → 11 binary | 25 | 34 |
| 4. Ordinals | Map 7 categoricals to integers | 34 | 34 |
| 5. One-hot | Expand 4 nominal cols → 17 binary | 34 | 47 |
| 6. Contributions | Expand 1 multi-select → 6 binary | 47 | 52 |
| 7. Missing values | Impute GPA median; drop >20% missing rows | 52 | 52 |
| 8. Normalize | Min-Max scale 16 non-binary numeric cols | 52 | 52 |
| 9. Save | Drop `course_code` → 50 feature cols | 52 | **50** |

**Final dataset: 31 rows × 50 features, 0 missing values**